## Import Packages and Datasets

In [ ]:
import os

import qcportal
from qcportal.singlepoint import SinglepointDriver
from qcportal.singlepoint import QCSpecification
from qcportal.optimization import OptimizationSpecification

from qcportal.external import scaffold

In [2]:
ADDRESS = "https://api.qcarchive.molssi.org:443"
client = qcportal.PortalClient(
    ADDRESS, 
    username=os.environ['QCARCHIVE_USER'],
    password=os.environ['QCARCHIVE_PASSWORD'],
    cache_dir=".",
)

## Get Dataset, Update Description

In [3]:
dataset_name = "TM Benchmark Optimization Dataset Step 2 v0.0"
description = (
    """
    This dataset includes single metal complexes with: {'Pd', 'Fe', 'Zn', 'Mg', 'Cu', 'Li'}, and the non-metals:
    {'C', 'H', 'P', 'S', 'O', 'N', 'F', 'Cl', 'Br'}, with a complex charge of {-1,0,+1}. Additionally, there are 
    some organic molecules for benchmarking purposes. All molecules are taken from those completed optimizations 
    in the optimization dataset "TM Benchmark Optimization Dataset Step 1 v0.0". These complexes are optimized 
    at a higher level of theory with mp3/def2-tzvppd or mp2/def2-qzvppd with and without frozen core. The 
    molecular weight min, mean, and max are 81, 421, and 1026, respectively. There are 163 unique molecules, 
    each transition metal complex is submitted with multiplicities that completed on the previous step to 
    assess the spin state.
    """
)
#dataset = scaffold.from_json("scaffold_sos-mp2_hf.json.bz2", client)
dataset = client.get_dataset("optimization", dataset_name)
dataset.description = description
dataset.tags = ["openff"]
dataset.extras["long_description"] = description
dataset.extras["submitter"] = "jaclark5"

## Delete Old Specification

In [4]:
print(f"There are {len(dataset.specifications)} specifications")
for spec_name in dataset.specification_names:
    dataset.delete_specification(spec_name, delete_records=True)

There are 4 specifications


## Add New Specification

In [5]:
spec_types = [
    ["mp3", "def2-tzvppd", True],
    ["mp3", "def2-tzvppd", False],
    ["mp2", "def2-qzvppd", True],
    ["mp2", "def2-qzvppd", False],
]
for method, basis, fc in spec_types:
    spec = OptimizationSpecification(
        program='geometric',
        qc_specification=QCSpecification(
            program='psi4',
            driver=SinglepointDriver.deferred,
            method=method,
            basis=basis,
            keywords={
                'maxiter': 500, 
                'scf_properties': ['dipole', 'quadrupole', 'wiberg_lowdin_indices', 'mayer_indices', 
                                   'lowdin_charges', 'mulliken_charges'],
                'function_kwargs': {'properties': ['dipole_polarizabilities']},
                'properties_origin': ['COM'],
                'reference': 'uhf',
                "print": 3,
                "freeze_core": fc,
            },
        ),
        keywords={
            'tmax': 0.3,
            'check': 0,
            'qccnv': False,
            'reset': True,
            'trust': 0.1,
            'molcnv': False,
            'enforce': 0.0,
            'epsilon': 1e-05,
            'maxiter': 500,
            'coordsys': 'dlc',
            'constraints': {},
            'convergence_set': 'GAU',
        }, # keywords for geometric
    )
    label = "-fc" if fc else ""
    dataset.add_specification(name=f"{method}/{basis}{label}", specification=spec)

In [6]:
dataset.submit()
print(f"There are {dataset.record_count} records")
scaffold.to_json(dataset, compress=True, filename="scaffold_mp2_mp3.json")

There are 656 records


In [8]:
dataset.status()

{'mp2/def2-qzvppd-fc': {<RecordStatusEnum.waiting: 'waiting'>: 164},
 'mp3/def2-tzvppd-fc': {<RecordStatusEnum.waiting: 'waiting'>: 164},
 'mp2/def2-qzvppd': {<RecordStatusEnum.waiting: 'waiting'>: 164},
 'mp3/def2-tzvppd': {<RecordStatusEnum.waiting: 'waiting'>: 164}}